In [10]:
import pandas as pd, requests, time
from pathlib import Path
from io import StringIO

ROOT = Path("/home/py/groundwater/data/eccc")

In [3]:
OUT = ROOT / 'eccc'

In [30]:
BASE = "https://dd.weather.gc.ca/today/climate/observations"
PROV = {"ALBERTA": "AB", "BRITISH COLUMBIA": "BC", "SASKATCHEWAN": "SK",
        "MANITOBA": "MB", "ONTARIO": "ON", "QUEBEC": "QC",
        "NEW BRUNSWICK": "NB", "NOVA SCOTIA": "NS", "PRINCE EDWARD ISLAND": "PE",
        "NEWFOUNDLAND AND LABRADOR": "NL", "YUKON": "YT",
        "NORTHWEST TERRITORIES": "NT", "NUNAVUT": "NU"}

Download st from: https://climate.weather.gc.ca/map/index_e.html

In [31]:
st = pd.read_csv(ROOT / 'Station_search_results.csv')
st = st[st["Daily data first year"].notna() & st["Daily data last year"].notna()]
st["y0"] = pd.to_datetime(st["Daily data first year"]).dt.year
st["y1"] = pd.to_datetime(st["Daily data last year"]).dt.year
st['Station name'].values

array(['BANFF CS', 'BOW VALLEY', 'NAKISKA RIDGETOP'], dtype=object)

In [32]:
S = requests.Session()
S.headers.update({"User-Agent": "gwmo/0.1"})

In [33]:
for _, r in st.iterrows():
    cid, prov = r["Climate Identifier"], PROV[r["Province or territory"]]
    dest_dir = OUT / 'raw' / cid
    dest_dir.mkdir(exist_ok=True)

    for yr in range(r["y0"], r["y1"] + 1):
        fn = f"climate_daily_{prov}_{cid}_{yr}_P1D.csv"
        dest = dest_dir / fn
        if dest.exists():
            continue

        resp = S.get(f"{BASE}/daily/csv/{prov}/{fn}", timeout=30)
        if resp.status_code == 404:
            print(f"  missing {cid} {yr}")
            continue
        resp.raise_for_status()

        dest.write_text(resp.text)
        time.sleep(0.15)

    print(f"{r['Station name']} ({cid}) done")

BANFF CS (3050519) done
BOW VALLEY (3050778) done
NAKISKA RIDGETOP (305MGFF) done


In [45]:
VALUE_COLS = ["Max Temp (°C)", "Min Temp (°C)", "Mean Temp (°C)",
              "Heat Deg Days (°C)", "Cool Deg Days (°C)",
              "Total Rain (mm)", "Total Snow (cm)", "Total Precip (mm)",
              "Snow on Grnd (cm)", "Spd of Max Gust (km/h)"]

In [46]:
def clean_station(cid):
    files = sorted((OUT / 'raw' / cid).glob("*.csv"))
    if not files:
        return None
    df = pd.concat((pd.read_csv(f, dtype=str) for f in files), ignore_index=True)
    df["date"] = pd.to_datetime(df["Date/Time"])

    for c in VALUE_COLS:
        flag = c.split(" (")[0] + " Flag"
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[flag] == "T", c] = 0.0       # trace precip is ~0, not missing
        df.loc[df[flag] == "M", c] = pd.NA

    for c in ["Latitude (y)", "Longitude (x)"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = (df.rename(columns={"Latitude (y)": "lat", "Longitude (x)": "lon",
                             "Climate ID": "climate_id", "Station Name": "station"})
            [["climate_id", "station", "lat", "lon", "date"] + VALUE_COLS]
            .drop_duplicates(subset="date", keep="last")
            .sort_values("date")
            .reset_index(drop=True))

    # reindex onto a continuous daily axis so gaps are explicit NaN rows
    full = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = (df.set_index("date").reindex(full).rename_axis("date").reset_index())
    df[["climate_id", "station", "lat", "lon"]] = df[["climate_id", "station", "lat", "lon"]].ffill().bfill()
    return df

In [47]:
CLEAN = OUT / 'processed'

In [49]:
summary = []
for cid in st["Climate Identifier"]:
    df = clean_station(cid)
    if df is None:
        print(f"no raw files for {cid}")
        continue
    name = df["station"].iloc[0].replace(" ", "_")
    df.to_csv(CLEAN / f"{cid}_{name}.csv", index=False)
    summary.append({"climate_id": cid, "station": df["station"].iloc[0],
                    "start": df["date"].min(), "end": df["date"].max(),
                    "n_days": len(df),
                    "pct_missing_meantemp": df["Mean Temp (°C)"].isna().mean().round(3)})

pd.DataFrame(summary)

,climate_id,station,start,end,n_days,pct_missing_meantemp
0,3050519,BANFF CS,1995-01-01,2026-12-31,11688,0.024
1,3050778,BOW VALLEY,1993-01-01,2026-12-31,12418,0.055
2,305MGFF,NAKISKA RIDGETOP,1994-01-01,2026-12-31,12053,0.204
